In [7]:
#1
import json
from pathlib import Path

import chromadb

from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

from sentence_transformers import SentenceTransformer

In [10]:
#2
DOCUMENTS = Path("documents")

nvd_file = DOCUMENTS / "nvd_cve_2026_23066.json"
linux_file = DOCUMENTS / "linux_cna_CVE-2026-23066.json"
openwrt_file = DOCUMENTS / "openwrt-25.12.5-armsr-armv8.bom.cdx.json"
patch_file = DOCUMENTS / "linux_patch_CVE-2026-23066.patch"

print(nvd_file.exists())
print(linux_file.exists())
print(openwrt_file.exists())
print(patch_file.exists())

True
True
True
True


In [12]:
#3
with open(nvd_file, "r", encoding="utf-8") as f:
    nvd = json.load(f)

with open(linux_file, "r", encoding="utf-8") as f:
    linux_cna = json.load(f)

with open(openwrt_file, "r", encoding="utf-8") as f:
    openwrt = json.load(f)

patch = patch_file.read_text(encoding="utf-8")

print("NVD loaded")
print("Linux CNA loaded")
print("OpenWrt SBOM loaded")
print("Patch loaded")

NVD loaded
Linux CNA loaded
OpenWrt SBOM loaded
Patch loaded


In [13]:
#4
nvd_cve = nvd["vulnerabilities"][0]["cve"]

nvd_description = next(
    item["value"]
    for item in nvd_cve["descriptions"]
    if item["lang"] == "en"
)

nvd_text = f"""
SOURCE: NVD
CVE: {nvd_cve["id"]}

DESCRIPTION:
{nvd_description}

METRICS:
{json.dumps(nvd_cve.get("metrics", {}), indent=2)}

CONFIGURATIONS:
{json.dumps(nvd_cve.get("configurations", []), indent=2)}

REFERENCES:
{json.dumps(nvd_cve.get("references", []), indent=2)}
"""

print(nvd_text)


SOURCE: NVD
CVE: CVE-2026-23066

DESCRIPTION:
In the Linux kernel, the following vulnerability has been resolved:

rxrpc: Fix recvmsg() unconditional requeue

If rxrpc_recvmsg() fails because MSG_DONTWAIT was specified but the call at
the front of the recvmsg queue already has its mutex locked, it requeues
the call - whether or not the call is already queued.  The call may be on
the queue because MSG_PEEK was also passed and so the call was not dequeued
or because the I/O thread requeued it.

The unconditional requeue may then corrupt the recvmsg queue, leading to
things like UAFs or refcount underruns.

Fix this by only requeuing the call if it isn't already on the queue - and
moving it to the front if it is already queued.  If we don't queue it, we
have to put the ref we obtained by dequeuing it.

Also, MSG_PEEK doesn't dequeue the call so shouldn't call
rxrpc_notify_socket() for the call if we didn't use up all the data on the
queue, so fix that also.

METRICS:
{
  "cvssMetricV31":

In [14]:
#5
cna = linux_cna["containers"]["cna"]

linux_description = next(
    item["value"]
    for item in cna["descriptions"]
    if item["lang"] == "en"
)

linux_text = f"""
SOURCE: Linux Kernel CNA
CVE: CVE-2026-23066

TITLE:
{cna.get("title", "")}

DESCRIPTION:
{linux_description}

AFFECTED / FIXED VERSIONS:
{json.dumps(cna.get("affected", []), indent=2)}

REFERENCES AND FIX COMMITS:
{json.dumps(cna.get("references", []), indent=2)}
"""

print(linux_text)


SOURCE: Linux Kernel CNA
CVE: CVE-2026-23066

TITLE:
rxrpc: Fix recvmsg() unconditional requeue

DESCRIPTION:
In the Linux kernel, the following vulnerability has been resolved:

rxrpc: Fix recvmsg() unconditional requeue

If rxrpc_recvmsg() fails because MSG_DONTWAIT was specified but the call at
the front of the recvmsg queue already has its mutex locked, it requeues
the call - whether or not the call is already queued.  The call may be on
the queue because MSG_PEEK was also passed and so the call was not dequeued
or because the I/O thread requeued it.

The unconditional requeue may then corrupt the recvmsg queue, leading to
things like UAFs or refcount underruns.

Fix this by only requeuing the call if it isn't already on the queue - and
moving it to the front if it is already queued.  If we don't queue it, we
have to put the ref we obtained by dequeuing it.

Also, MSG_PEEK doesn't dequeue the call so shouldn't call
rxrpc_notify_socket() for the call if we didn't use up all the dat

In [15]:
#6
kernel_components = [
    component
    for component in openwrt["components"]
    if component.get("name", "").lower() == "kernel"
]

print(f"Kernel components found: {len(kernel_components)}")

print(
    json.dumps(
        kernel_components,
        indent=2
    )
)

Kernel components found: 1
[
  {
    "name": "kernel",
    "type": "operating-system",
    "licenses": [
      {
        "license": {
          "name": "GPL-2.0"
        }
      }
    ],
    "cpe": "cpe:/o:linux:linux_kernel:6.12.94",
    "version": "6.12.94"
  }
]


In [16]:
#7
openwrt_text = f"""
SOURCE: Official OpenWrt CycloneDX SBOM
PRODUCT: OpenWrt 25.12.5
TARGET: armsr/armv8
CVE UNDER INVESTIGATION: CVE-2026-23066

RELEVANT COMPONENT:

{json.dumps(kernel_components, indent=2)}
"""

print(openwrt_text)



SOURCE: Official OpenWrt CycloneDX SBOM
PRODUCT: OpenWrt 25.12.5
TARGET: armsr/armv8
CVE UNDER INVESTIGATION: CVE-2026-23066

RELEVANT COMPONENT:

[
  {
    "name": "kernel",
    "type": "operating-system",
    "licenses": [
      {
        "license": {
          "name": "GPL-2.0"
        }
      }
    ],
    "cpe": "cpe:/o:linux:linux_kernel:6.12.94",
    "version": "6.12.94"
  }
]



In [17]:
#8
patch_text = f"""
SOURCE: Linux Stable Kernel Patch
CVE: CVE-2026-23066

{patch}
"""

print(patch_text[:3000])


SOURCE: Linux Stable Kernel Patch
CVE: CVE-2026-23066

From cf969bddd6e69c5777fa89dc88402204e72f312a Mon Sep 17 00:00:00 2001
From: David Howells <dhowells@redhat.com>
Date: Thu, 26 Feb 2026 16:23:46 +0800
Subject: rxrpc: Fix recvmsg() unconditional requeue

[ Upstream commit 2c28769a51deb6022d7fbd499987e237a01dd63a ]

If rxrpc_recvmsg() fails because MSG_DONTWAIT was specified but the call at
the front of the recvmsg queue already has its mutex locked, it requeues
the call - whether or not the call is already queued.  The call may be on
the queue because MSG_PEEK was also passed and so the call was not dequeued
or because the I/O thread requeued it.

The unconditional requeue may then corrupt the recvmsg queue, leading to
things like UAFs or refcount underruns.

Fix this by only requeuing the call if it isn't already on the queue - and
moving it to the front if it is already queued.  If we don't queue it, we
have to put the ref we obtained by dequeuing it.

Also, MSG_PEEK doesn't deq

In [18]:
#9
documents = [
    Document(
        text=nvd_text,
        metadata={
            "source": "NVD",
            "cve": "CVE-2026-23066"
        }
    ),

    Document(
        text=linux_text,
        metadata={
            "source": "Linux CNA",
            "cve": "CVE-2026-23066"
        }
    ),

    Document(
        text=openwrt_text,
        metadata={
            "source": "OpenWrt SBOM",
            "cve": "CVE-2026-23066"
        }
    ),

    Document(
        text=patch_text,
        metadata={
            "source": "Linux patch",
            "cve": "CVE-2026-23066"
        }
    )
]

print(f"Documents prepared: {len(documents)}")

Documents prepared: 4


In [19]:
#10
splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50
)

nodes = splitter.get_nodes_from_documents(documents)

print(f"Number of chunks: {len(nodes)}")

Number of chunks: 14


In [20]:
#11
for i, node in enumerate(nodes):
    print("=" * 80)
    print(f"CHUNK {i}")
    print(f"Source: {node.metadata['source']}")
    print()
    print(node.get_content()[:1000])
    print()

CHUNK 0
Source: NVD

SOURCE: NVD
CVE: CVE-2026-23066

DESCRIPTION:
In the Linux kernel, the following vulnerability has been resolved:

rxrpc: Fix recvmsg() unconditional requeue

If rxrpc_recvmsg() fails because MSG_DONTWAIT was specified but the call at
the front of the recvmsg queue already has its mutex locked, it requeues
the call - whether or not the call is already queued.  The call may be on
the queue because MSG_PEEK was also passed and so the call was not dequeued
or because the I/O thread requeued it.

The unconditional requeue may then corrupt the recvmsg queue, leading to
things like UAFs or refcount underruns.

Fix this by only requeuing the call if it isn't already on the queue - and
moving it to the front if it is already queued.  If we don't queue it, we
have to put the ref we obtained by dequeuing it.

Also, MSG_PEEK doesn't dequeue the call so shouldn't call
rxrpc_notify_socket() for the call if we didn't use up all the data on the
queue, so fix that also.

METRICS:


In [21]:
#12
embedding_model = SentenceTransformer(
    "sentence-transformers/multi-qa-mpnet-base-cos-v1"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [22]:
#12
chunk_texts = [
    node.get_content()
    for node in nodes
]

In [23]:
#13
embeddings = embedding_model.encode_document(
    chunk_texts,
    normalize_embeddings=True
)

print("Number of embeddings:", len(embeddings))
print("Dimensions per embedding:", embeddings.shape[1])

Number of embeddings: 14
Dimensions per embedding: 768


In [24]:
#14
client = chromadb.PersistentClient(
    path="./chroma_db"
)

print("Chroma client created.")

Chroma client created.


In [26]:
#15
collection_name = "cve_2026_23066"

existing_collections = [
    collection.name
    for collection in client.list_collections()
]

if collection_name in existing_collections:
    client.delete_collection(collection_name)

collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=None
)

print("Collection ready.")

Collection ready.


In [27]:
#16
ids = [
    f"chunk_{i}"
    for i in range(len(nodes))
]

metadata = [
    {
        "source": node.metadata["source"],
        "cve": node.metadata["cve"]
    }
    for node in nodes
]

In [28]:
#17
collection.add(
    ids=ids,
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=metadata
)

print(
    "Chunks stored in Chroma:",
    collection.count()
)

Chunks stored in Chroma: 14


In [29]:
#18
query = """
Does CVE-2026-23066 appear to affect the Linux kernel
6.12.94 contained in OpenWrt 25.12.5?

Retrieve evidence concerning:
- affected versions
- fixed versions
- installed OpenWrt kernel version
- relevant fix or patch
"""

print(query)


Does CVE-2026-23066 appear to affect the Linux kernel
6.12.94 contained in OpenWrt 25.12.5?

Retrieve evidence concerning:
- affected versions
- fixed versions
- installed OpenWrt kernel version
- relevant fix or patch



In [30]:
#19
query_embedding = embedding_model.encode_query(
    query,
    normalize_embeddings=True
)

print(query_embedding.shape)

(768,)


In [31]:
#20
results = collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=5,
    where={
        "cve": "CVE-2026-23066"
    }
)

In [32]:
#21
for rank, (text, meta) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0]
    ),
    start=1
):
    print("=" * 80)
    print(f"RANK {rank}")
    print(f"Source: {meta['source']}")
    print()
    print(text)
    print()

RANK 1
Source: OpenWrt SBOM

SOURCE: Official OpenWrt CycloneDX SBOM
PRODUCT: OpenWrt 25.12.5
TARGET: armsr/armv8
CVE UNDER INVESTIGATION: CVE-2026-23066

RELEVANT COMPONENT:

[
  {
    "name": "kernel",
    "type": "operating-system",
    "licenses": [
      {
        "license": {
          "name": "GPL-2.0"
        }
      }
    ],
    "cpe": "cpe:/o:linux:linux_kernel:6.12.94",
    "version": "6.12.94"
  }
]

RANK 2
Source: NVD

3:o:linux:linux_kernel:*:*:*:*:*:*:*:*",
            "versionStartIncluding": "4.11",
            "versionEndExcluding": "6.18.8",
            "matchCriteriaId": "A85A6630-48F5-47FB-9871-DB4D2269F664"
          },
          {
            "vulnerable": true,
            "criteria": "cpe:2.3:o:linux:linux_kernel:6.19:rc1:*:*:*:*:*:*",
            "matchCriteriaId": "17B67AA7-40D6-4AFA-8459-F200F3D7CFD1"
          },
          {
            "vulnerable": true,
            "criteria": "cpe:2.3:o:linux:linux_kernel:6.19:rc2:*:*:*:*:*:*",
            "matchCriteri

In [33]:
#22
retrieved_chunks = results["documents"][0]
retrieved_metadata = results["metadatas"][0]

evidence = ""

for i, (text, meta) in enumerate(
    zip(retrieved_chunks, retrieved_metadata),
    start=1
):
    evidence += f"""
EVIDENCE {i}
Source: {meta['source']}

{text}

-----------------------------
"""

In [34]:
#23
prompt = f"""
You are assisting a product-security analyst performing
vulnerability triage.

Candidate vulnerability:
CVE-2026-23066

Product:
OpenWrt 25.12.5 armsr/armv8

Component:
Linux kernel 6.12.94

QUESTION:

Does CVE-2026-23066 appear to affect this product?

Using ONLY the evidence supplied below, return:

Assessment:
AFFECTED / NOT AFFECTED / INSUFFICIENT EVIDENCE

Supporting evidence:

Patch/fix evidence:

Contradictory evidence:

Missing information or uncertainty:

Sources:

Do not make claims that are unsupported by the supplied evidence.

RETRIEVED EVIDENCE:

{evidence}
"""

print(prompt)


You are assisting a product-security analyst performing
vulnerability triage.

Candidate vulnerability:
CVE-2026-23066

Product:
OpenWrt 25.12.5 armsr/armv8

Component:
Linux kernel 6.12.94

QUESTION:

Does CVE-2026-23066 appear to affect this product?

Using ONLY the evidence supplied below, return:

Assessment:
AFFECTED / NOT AFFECTED / INSUFFICIENT EVIDENCE

Supporting evidence:

Patch/fix evidence:

Contradictory evidence:

Missing information or uncertainty:

Sources:

Do not make claims that are unsupported by the supplied evidence.

RETRIEVED EVIDENCE:


EVIDENCE 1
Source: OpenWrt SBOM

SOURCE: Official OpenWrt CycloneDX SBOM
PRODUCT: OpenWrt 25.12.5
TARGET: armsr/armv8
CVE UNDER INVESTIGATION: CVE-2026-23066

RELEVANT COMPONENT:

[
  {
    "name": "kernel",
    "type": "operating-system",
    "licenses": [
      {
        "license": {
          "name": "GPL-2.0"
        }
      }
    ],
    "cpe": "cpe:/o:linux:linux_kernel:6.12.94",
    "version": "6.12.94"
  }
]

-----------